# AI-Powered Automatic Retail Checkout System
## YOLOv8-seg Training on MVTec D2S Dataset
**Authors:** Varisha & Muzzammil | **Course:** Deep Neural Networks

This notebook trains a YOLOv8-seg instance segmentation model on the MVTec D2S dataset (60 grocery categories) using Google Colab GPU.


## 1. Mount Google Drive & Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Set your dataset path in Google Drive
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/dnn_project"
D2S_ROOT = f"{DRIVE_PROJECT_DIR}/mvtec_d2s"

import os
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
print(f"Project directory: {DRIVE_PROJECT_DIR}")
print(f"D2S dataset root: {D2S_ROOT}")
print(f"Dataset exists: {os.path.exists(D2S_ROOT)}")


## 2. Install Dependencies

In [ ]:
!pip install -q ultralytics>=8.0.0 pycocotools albumentations PyYAML tqdm seaborn

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")


## 3. Upload Project Files
Upload the project files from your local machine or clone from Drive.

In [ ]:
# Copy project files from Drive to Colab workspace
WORK_DIR = "/content/ai_checkout"
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

# Copy scripts from Drive if they exist
import shutil
for f in ["data_preparation.py", "train.py", "validate.py", "predict.py", "d2s.yaml"]:
    src = f"{DRIVE_PROJECT_DIR}/{f}"
    if os.path.exists(src):
        shutil.copy2(src, f"{WORK_DIR}/{f}")
        print(f"Copied {f}")
    else:
        print(f"Not found in Drive: {f}")

print(f"\nWorking directory: {os.getcwd()}")
print(f"Contents: {os.listdir(WORK_DIR)}")


## 4. Prepare Dataset (COCO to YOLO Format)

In [ ]:
# Convert MVTec D2S from COCO JSON to YOLO segmentation format
!python data_preparation.py --d2s-root "{D2S_ROOT}" --output ./mvtec_d2s_yolo --yaml-ref ./d2s.yaml

# Verify output
print("\nOutput structure:")
for root, dirs, files in os.walk("./mvtec_d2s_yolo"):
    level = root.replace("./mvtec_d2s_yolo", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        for f in files[:5]:
            print(f"{indent}  {f}")
        if len(files) > 5:
            print(f"{indent}  ... and {len(files) - 5} more files")


## 5. Train YOLOv8-seg

In [ ]:
# Train YOLOv8n-seg (nano) on MVTec D2S
# Adjust batch size based on available GPU memory:
#   - T4 (16GB):  batch=16
#   - A100 (40GB): batch=32
#   - V100 (16GB): batch=16

BATCH_SIZE = 16  # Reduce to 8 if you get OOM errors
EPOCHS = 100
MODEL_SIZE = "n"  # n=nano, s=small, m=medium

!python train.py \
    --model yolo \
    --data ./mvtec_d2s_yolo/d2s.yaml \
    --epochs {EPOCHS} \
    --batch {BATCH_SIZE} \
    --size {MODEL_SIZE} \
    --device 0

print("\nTraining complete!")


## 6. Validate Model

In [ ]:
# Run full COCO evaluation on the validation set
!python validate.py \
    --model yolo \
    --weights runs/segment/train/weights/best.pt \
    --data ./mvtec_d2s_yolo/d2s.yaml \
    --device 0


## 7. Test Prediction

In [ ]:
# Find a validation image to test
val_dir = f"{D2S_ROOT}/images/val"
val_images = [f for f in os.listdir(val_dir) if f.endswith((".jpg", ".png"))]
test_image = os.path.join(val_dir, val_images[0]) if val_images else None

if test_image:
    print(f"Testing with: {test_image}")
    !python predict.py --image "{test_image}" --model yolo --weights runs/segment/train/weights/best.pt --output test_result.jpg

    # Display result
    from IPython.display import Image, display
    display(Image("test_result.jpg", width=800))
else:
    print("No validation images found.")


## 8. Save Weights to Google Drive

In [ ]:
# Copy trained weights and results back to Google Drive
save_dir = f"{DRIVE_PROJECT_DIR}/trained_weights"
os.makedirs(save_dir, exist_ok=True)

files_to_save = [
    ("runs/segment/train/weights/best.pt", "best.pt"),
    ("runs/segment/train/weights/last.pt", "last.pt"),
    ("runs/segment/train/results.csv", "results.csv"),
    ("runs/segment/train/training_curves.png", "training_curves.png"),
]

for src, dst in files_to_save:
    if os.path.exists(src):
        shutil.copy2(src, f"{save_dir}/{dst}")
        print(f"Saved {dst} to Drive")
    else:
        print(f"Not found: {src}")

print(f"\nWeights saved to: {save_dir}")
print("Copy best.pt to your backend/weights/ directory for inference.")


## 9. Training Plots

In [ ]:
# Display training result plots
from IPython.display import Image, display
import glob

plot_files = glob.glob("runs/segment/train/*.png")
for pf in sorted(plot_files):
    print(f"\n{os.path.basename(pf)}")
    display(Image(pf, width=800))
